In [53]:
"""
Weather Fire Risk Modeling (deep learning only: DNN + TabNet)
================================================================
Modular pipeline to predict very-low/low/moderate/high wildfire using weather parameters 
using only two PyTorch-based models:
  load_data()        -> reads the CSV
  split_data()       -> 70/30 train-test split for a given target
  train_dnn()        -> a simple feedforward neural network (MLP) in PyTorch
  train_tabnet()     -> TabNet (via the pytorch-tabnet package)
  test_model()       -> runs predictions, returns a DataFrame of
                               actual vs. predicted values
  evaluate_models()  -> computes metrics for each model's results
  compare_models()   -> prints/returns a comparison table
  main()             -> wires everything together (No separate main function in Jupyter notebook)
 
Both train_dnn() and train_tabnet() are wrapped so they expose a plain .predict() interface, 
which means test_model(), evaluate_models(), and compare_models() work the same way
regardless of model type.
 
LATITUDE, LONGITUDE, NEAREST_WEATHER_LAT, and NEAREST_WEATHER_LON are
excluded from model training, as is DISCOVERYDATETIME (a
raw timestamp string that isn't usable as a numeric feature as-is).
These columns are NOT dropped from the data entirely -- their test-split
values are carried through and attached to results_df alongside the
Actual/Predicted columns, so you can still see them for reference.
 
Requires:
    pip/conda install torch pytorch-tabnet pandas numpy scikit-learn
 
NOTE ON DATA LEAKAGE:
WEATHER_RISK_CATEGORY is a deterministic binning of WEATHER_FIRE_RISK_SCORE
(High = 60.01-79.99, Moderate = 40.01-60.00, Low = 21.10-40.00,
Very Low = 15.00-20.00). Each target is excluded from the other task's
input columns so a model can't just "read off" the answer.
"""

'\nWeather Fire Risk Modeling (deep learning only: DNN + TabNet)\n================================================================\nModular pipeline to predict very-low/low/moderate/high wildfire using weather parameters \nusing only two PyTorch-based models:\n  load_data()        -> reads the CSV\n  split_data()       -> 70/30 train-test split for a given target\n  train_dnn()        -> a simple feedforward neural network (MLP) in PyTorch\n  train_tabnet()     -> TabNet (via the pytorch-tabnet package)\n  test_model()       -> runs predictions, returns a DataFrame of\n                               actual vs. predicted values\n  evaluate_models()  -> computes metrics for each model\'s results\n  compare_models()   -> prints/returns a comparison table\n  main()             -> wires everything together (No separate main function in Jupyter notebook)\n\nBoth train_dnn() and train_tabnet() are wrapped so they expose a plain .predict() interface, \nwhich means test_model(), evaluate_models

In [50]:
import warnings
warnings.filterwarnings("ignore")
 
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    r2_score,
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
)
import folium
from IPython.display import display, IFrame

#Check if a software/library is installed or not
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
 
try:
    from pytorch_tabnet.tab_model import TabNetClassifier, TabNetRegressor
    TABNET_AVAILABLE = True
except ImportError:
    TABNET_AVAILABLE = False

In [42]:
"""
Target location: Fairbanks and max_radius to consider
"""
TARGET_LAT = 64.8378 #north
TARGET_LON = -147.7164 #West
MAX_RADIUS = 250 #miles

In [5]:
RANDOM_STATE = 42
DATA_PATH = "filtered_training_data_selectedColumn_Category_together.csv"
CLASSIFICATION_TARGET = "WEATHER_RISK_CATEGORY"
REGRESSION_TARGET = "WEATHER_FIRE_RISK_SCORE"

In [1]:
# Columns to exclude from model training but keep tracked in the results
# (LATITUDE/LONGITUDE/NEAREST_WEATHER_LAT/NEAREST_WEATHER_LON per request;
# DISCOVERYDATETIME added because it's a raw timestamp string that breaks
# training as-is -- see note in the response).
EXCLUDE_COLS = [
   'LATITUDE', 'LONGITUDE', 'NEAREST_WEATHER_LAT', 'NEAREST_WEATHER_LON', 'DISCOVERYDATETIME',  
   #'avg_max_temp_7d', 'avg_min_temp_7d', 'avg_rh_7d', 'total_precip_7d', 'max_wind_7d', 'consec_dry_days_7d', 'avg_solar_7d', 
   #'max_temp_7d', 'min_rh_7d', 'max_solar_7d',
   #'avg_max_temp_30d', 'total_precip_30d', 'precip_anomaly_30d', 'avg_wind_30d', 'avg_rh_30d', 'days_no_rain_30d', 'avg_solar_30d', 
   #'avg_max_temp_90d', 'max_temp_90d', 'total_precip_90d', 'days_no_rain_90d', 'avg_rh_90d', 'total_solar_90d', 
   #'days_above_heat_threshold', 'days_above_wind_threshold', 'days_low_rh', 
   #'IGNITION_WEATHER_SCORE', 'SEASONAL_SCORE', 'ASPECT_SCORE', 'ELEVATION_SCORE', 'SLOPE_WIND_SCORE', 'SPREAD_RATE_SCORE',
   'WEATHER_FIRE_RISK_SCORE', 
   'WEATHER_RISK_CATEGORY'
]
# Note The commented out columns will remain in the dataframe

In [7]:
if TORCH_AVAILABLE:
    class SimpleMLP(nn.Module):
        """A small feedforward network: input -> 128 -> 64 -> output."""
 
        def __init__(self, input_dim, output_dim, task_type):
            super().__init__()
            self.task_type = task_type
            self.net = nn.Sequential(
                nn.Linear(input_dim, 128),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(64, output_dim),
            )
 
        def forward(self, x):
            return self.net(x)

In [9]:
#----------------------------------------------------------------------#
# Wrapper classes
#----------------------------------------------------------------------#

class DNNWrapper:
    """Wraps a trained PyTorch MLP so .predict() takes a pandas/NumPy
    array and returns labels/values in the original space."""
 
    def __init__(self, model, scaler, task_type, label_encoder=None):
        self.model = model
        self.scaler = scaler
        self.task_type = task_type
        self.label_encoder = label_encoder
 
    def predict(self, X):
        X_np = X.values if hasattr(X, "values") else np.asarray(X)
        X_scaled = self.scaler.transform(X_np)
        X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
 
        self.model.eval()
        with torch.no_grad():
            outputs = self.model(X_tensor)
 
        if self.task_type == "classification":
            preds = outputs.argmax(dim=1).numpy()
            if self.label_encoder is not None:
                preds = self.label_encoder.inverse_transform(preds)
            return preds
        else:
            return outputs.squeeze(-1).numpy()

class TabNetWrapper:
    """Wraps a trained TabNet model so .predict() takes a pandas/NumPy
    array and returns labels/values in the original space."""
 
    def __init__(self, model, task_type, label_encoder=None):
        self.model = model
        self.task_type = task_type
        self.label_encoder = label_encoder
 
    def predict(self, X):
        X_np = X.values if hasattr(X, "values") else np.asarray(X)
        X_np = X_np.astype(np.float32)
        preds = self.model.predict(X_np)
 
        if self.task_type == "classification":
            preds = preds.astype(int).ravel()
            if self.label_encoder is not None:
                preds = self.label_encoder.inverse_transform(preds)
            return preds
        else:
            return preds.ravel()

In [10]:
# ------------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------------
def load_data(path):
    """Reads the CSV and returns a single DataFrame."""
    df = pd.read_csv(path)
    return df

In [11]:
# ------------------------------------------------------------------
# 2. Split data
# ------------------------------------------------------------------
def split_data(df, target_col, task_type, exclude_cols=None, test_size=0.30, random_state=RANDOM_STATE):
    """Splits a dataframe into 70% train / 30% test for the given target
    column. Stratifies on the target for classification tasks.
 
    exclude_cols: columns that should NOT be used as model inputs, but
    whose test-split values are still returned (as `tracking_test`) so
    they can be reattached to the results later.
    """
    exclude_cols = [c for c in (exclude_cols or []) if c in df.columns]
 
    X_full = df.drop(columns=[target_col])  # still includes exclude_cols
    y = df[target_col]
 
    if task_type == "classification":
        try:
            X_train_full, X_test_full, y_train, y_test = train_test_split(
                X_full, y, test_size=test_size, random_state=random_state, stratify=y
            )
        except ValueError:
            print("Warning: could not stratify (a class has too few "
                  "samples). Using a plain random split.")
            X_train_full, X_test_full, y_train, y_test = train_test_split(
                X_full, y, test_size=test_size, random_state=random_state
            )
    else:
        X_train_full, X_test_full, y_train, y_test = train_test_split(
            X_full, y, test_size=test_size, random_state=random_state
        )
 
    # Keep the excluded columns' test-split values for tracking purposes
    tracking_test = X_test_full[exclude_cols].reset_index(drop=True) if exclude_cols else None
 
    # Actual model inputs -- excluded columns dropped
    X_train = X_train_full.drop(columns=exclude_cols)
    X_test = X_test_full.drop(columns=exclude_cols)
 
    return X_train, X_test, y_train, y_test, tracking_test

In [12]:
# ------------------------------------------------------------------
# 3. Model training functions (each takes the training split as input)
# ------------------------------------------------------------------
def train_dnn(X_train, y_train, task_type, epochs=100, batch_size=64, lr=1e-3):
    """Trains a simple feedforward neural network (MLP) in PyTorch.
    Returns None if torch is not installed."""
    if not TORCH_AVAILABLE:
        print("NOTE: torch is not installed (pip install torch) -- "
              "skipping DNN model.")
        return None
 
    torch.manual_seed(RANDOM_STATE)
 
    X_train_np = X_train.values if hasattr(X_train, "values") else np.asarray(X_train)
    scaler = StandardScaler().fit(X_train_np)
    X_scaled = scaler.transform(X_train_np)
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
 
    label_encoder = None
    if task_type == "classification":
        label_encoder = LabelEncoder()
        y_enc = label_encoder.fit_transform(y_train)
        output_dim = len(label_encoder.classes_)
        y_tensor = torch.tensor(y_enc, dtype=torch.long)
        criterion = nn.CrossEntropyLoss()
    else:
        output_dim = 1
        y_values = y_train.values if hasattr(y_train, "values") else np.asarray(y_train)
        y_tensor = torch.tensor(y_values, dtype=torch.float32).view(-1, 1)
        criterion = nn.MSELoss()
 
    model = SimpleMLP(input_dim=X_tensor.shape[1], output_dim=output_dim, task_type=task_type)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
 
    dataset = TensorDataset(X_tensor, y_tensor)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
 
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0.0
        for xb, yb in loader:
            optimizer.zero_grad()
            outputs = model(xb)
            loss = criterion(outputs, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * xb.size(0)
 
        if (epoch + 1) % 20 == 0:
            print(f"  DNN [{task_type}] epoch {epoch + 1}/{epochs} "
                  f"- loss: {epoch_loss / len(dataset):.4f}")
 
    return DNNWrapper(model, scaler, task_type, label_encoder)
 
 
def train_tabnet(X_train, y_train, task_type, max_epochs=100, patience=15, batch_size=256):
    """Trains a TabNet model via pytorch-tabnet. Returns None if the
    package is not installed."""
    if not TABNET_AVAILABLE:
        print("NOTE: pytorch-tabnet is not installed "
              "(pip install pytorch-tabnet) -- skipping TabNet model.")
        return None
 
    X_train_np = X_train.values if hasattr(X_train, "values") else np.asarray(X_train)
    X_train_np = X_train_np.astype(np.float32)
 
    if task_type == "classification":
        label_encoder = LabelEncoder()
        y_enc = label_encoder.fit_transform(y_train)
        model = TabNetClassifier(seed=RANDOM_STATE, verbose=0)
        model.fit(
            X_train_np, y_enc,
            max_epochs=max_epochs, patience=patience, batch_size=batch_size,
        )
        return TabNetWrapper(model, task_type, label_encoder)
    else:
        y_values = y_train.values if hasattr(y_train, "values") else np.asarray(y_train)
        y_train_np = y_values.reshape(-1, 1).astype(np.float32)
        model = TabNetRegressor(seed=RANDOM_STATE, verbose=0)
        model.fit(
            X_train_np, y_train_np,
            max_epochs=max_epochs, patience=patience, batch_size=batch_size,
        )
        return TabNetWrapper(model, task_type)

In [13]:
# ------------------------------------------------------------------
# 4. Test model: predict on the test split, store actual vs. predicted
# ------------------------------------------------------------------
def test_model(model, X_test, y_test, model_name, tracking_df=None):
    """Runs the model on the test split and returns a DataFrame with the
    actual and predicted values side by side. If tracking_df is provided
    (e.g. LATITUDE/LONGITUDE/etc. that were excluded from training), its
    columns are attached alongside Actual/Predicted for reference."""
    y_pred = model.predict(X_test)
    results_df = pd.DataFrame({
        "Actual": y_test.reset_index(drop=True),
        "Predicted": y_pred,
    })
    results_df["Model"] = model_name
 
    if tracking_df is not None:
        results_df = pd.concat(
            [tracking_df.reset_index(drop=True), results_df], axis=1
        )
 
    return results_df

In [46]:
# ----------------------------------------------------------------------------------
# Helper plot function (takes a dataframe and plot it on a map based on the lat,lon) 
# ----------------------------------------------------------------------------------
def create_map(df, lat_col='LATITUDE', lon_col='LONGITUDE', zoom_start=1,
               center_lat=TARGET_LAT, center_lon=TARGET_LON,
                save_path=None, show_in_notebook=True):
    """
    Create an interactive Folium map with red circle markers for each row
    in the dataframe. Hovering over a circle shows a tooltip with all
    column headers and their corresponding values for that record.

    Parameters:
        df (pd.DataFrame): DataFrame containing at least LATITUDE and LONGITUDE columns.
        lat_col (str): Name of the latitude column.
        lon_col (str): Name of the longitude column.
        zoom_start (int): Initial zoom level of the map.
        save_path (str or None): If provided, saves the map to this HTML file path.
        show_in_notebook (bool): If True, displays the map inline in a Jupyter notebook.

    Returns:
        folium.Map: The generated map object.
    """
    # Drop rows with missing coordinates
    df = df.dropna(subset=[lat_col, lon_col])

    # Center the map on the mean lat/lon
    center_lat = df[lat_col].mean()
    center_lon = df[lon_col].mean()

    m = folium.Map(location=[center_lat, center_lon], zoom_start=zoom_start)

    for _, row in df.iterrows():
        # Build an HTML table of all column headers and values for this row
        rows_html = "".join(
            f"<tr><td style='padding:2px 6px;font-weight:bold;'>{col}</td>"
            f"<td style='padding:2px 6px;'>{row[col]}</td></tr>"
            for col in df.columns
        )
        tooltip_html = f"<table>{rows_html}</table>"

        folium.CircleMarker(
            location=[row[lat_col], row[lon_col]],
            radius=2,
            color='red',
            fill=True,
            fill_color='red',
            fill_opacity=0.8,
            tooltip=folium.Tooltip(tooltip_html, sticky=True)
        ).add_to(m)

    # Optionally save to an HTML file
    if save_path:
        m.save(save_path)

    # Optionally display inline in a Jupyter notebook
    if show_in_notebook:
        display(m)

    return m

In [14]:
# ------------------------------------------------------------------
# 5. Evaluate models: compute metrics from each model's results DataFrame
# ------------------------------------------------------------------
def evaluate_models(results_dict, task_type):
    """Takes a dict of {model_name: results_dataframe} (as produced by
    test_model) and returns a comparison DataFrame of evaluation metrics."""
    if not results_dict:
        print("No models were trained (missing torch / pytorch-tabnet?). "
              "Install the required packages and re-run.")
        return pd.DataFrame()
 
    rows = []
 
    for model_name, results_df in results_dict.items():
        y_true = results_df["Actual"]
        y_pred = results_df["Predicted"]
 
        if task_type == "classification":
            rows.append({
                "Model": model_name,
                "Accuracy (%)": accuracy_score(y_true, y_pred) * 100,
                "Precision (macro)": precision_score(y_true, y_pred, average="macro", zero_division=0),
                "Recall (macro)": recall_score(y_true, y_pred, average="macro", zero_division=0),
                "F1-score (macro)": f1_score(y_true, y_pred, average="macro", zero_division=0),
                "Precision (weighted)": precision_score(y_true, y_pred, average="weighted", zero_division=0),
                "Recall (weighted)": recall_score(y_true, y_pred, average="weighted", zero_division=0),
                "F1-score (weighted)": f1_score(y_true, y_pred, average="weighted", zero_division=0),
            })
        else:
            mape = mean_absolute_percentage_error(y_true, y_pred)
            rows.append({
                "Model": model_name,
                "R2": r2_score(y_true, y_pred),
                "MAE": mean_absolute_error(y_true, y_pred),
                "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
                "Percentage Accuracy (100-MAPE)": (1 - mape) * 100,
            })
 
    comparison_df = pd.DataFrame(rows).set_index("Model").round(4)
    return comparison_df

In [15]:
# ------------------------------------------------------------------
# 6. Compare models: display (and optionally save) the comparison table
# ------------------------------------------------------------------
def compare_models(comparison_df, title, save_path=None):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)
    if comparison_df.empty:
        print("(nothing to compare)")
        return comparison_df
    print(comparison_df)
    if save_path:
        comparison_df.to_csv(save_path)
        print(f"Saved comparison table to {save_path}")
    return comparison_df

In [16]:
# --- Load data ---
df = load_data(DATA_PATH)

# Exclude each target from the other task's inputs (see leakage note above)
df_for_classification = df.drop(columns=[REGRESSION_TARGET])
df_for_regression = df.drop(columns=[CLASSIFICATION_TARGET])

In [51]:
# --- Split data ---
Xc_train, Xc_test, yc_train, yc_test, tracking_c_test = split_data(
    df_for_classification, CLASSIFICATION_TARGET, task_type="classification",
    exclude_cols=EXCLUDE_COLS,
)
Xr_train, Xr_test, yr_train, yr_test, tracking_r_test = split_data(
    df_for_regression, REGRESSION_TARGET, task_type="regression",
    exclude_cols=EXCLUDE_COLS,
)

In [18]:
# --- Train models: classification ---
dnn_clf = train_dnn(Xc_train, yc_train, task_type="classification")
tabnet_clf = train_tabnet(Xc_train, yc_train, task_type="classification")

  DNN [classification] epoch 20/100 - loss: 0.0402
  DNN [classification] epoch 40/100 - loss: 0.0176
  DNN [classification] epoch 60/100 - loss: 0.0129
  DNN [classification] epoch 80/100 - loss: 0.0162
  DNN [classification] epoch 100/100 - loss: 0.0095


In [ ]:
# --- Test models: classification ---
clf_models = {
    "DNN": dnn_clf,
    "TabNet": tabnet_clf,
}
clf_results = {
    name: test_model(model, Xc_test, yc_test, name, tracking_df=tracking_c_test)
    for name, model in clf_models.items() if model is not None
}

#Save the results into a csv file
clf_results['DNN'].to_csv('clf_results_dnn.csv', index=False)
clf_results['TabNet'].to_csv('clf_results_tabnet.csv', index=False)

#Plot the data for easy visualization
m = create_map(clf_results['DNN'], save_path='dnn_predicted_vs_actual.html', zoom_start=3)
m = create_map(clf_results['TabNet'], save_path='tabnet_predicted_vs_actual.html', zoom_start=3)

In [ ]:
# --- Evaluate + compare: classification ---
clf_comparison = evaluate_models(clf_results, task_type="classification")
compare_models(
    clf_comparison,
    "CLASSIFICATION MODEL COMPARISON (target: WEATHER_RISK_CATEGORY)",
    save_path="dnn_classification_model_comparison_DNN.csv",
)

In [21]:
# --- Train models: regression ---
dnn_reg = train_dnn(Xr_train, yr_train, task_type="regression")
tabnet_reg = train_tabnet(Xr_train, yr_train, task_type="regression")

  DNN [regression] epoch 20/100 - loss: 22.6014
  DNN [regression] epoch 40/100 - loss: 19.2763
  DNN [regression] epoch 60/100 - loss: 17.2944
  DNN [regression] epoch 80/100 - loss: 14.8103
  DNN [regression] epoch 100/100 - loss: 13.4966


In [21]:
# --- Test models: regression ---
reg_models = {
    "DNN": dnn_reg,
    "TabNet": tabnet_reg,
}
reg_results = {
    name: test_model(model, Xr_test, yr_test, name, tracking_df=tracking_r_test)
    for name, model in reg_models.items() if model is not None
}

#Save the results into a csv file
reg_results['DNN'].to_csv('reg_results_dnn.csv', index=False)
reg_results['TabNet'].to_csv('reg_results_tabnet.csv', index=False)

#Plot the data for easy visualization
m = create_map(reg_results['DNN'], save_path='dnn_predicted_vs_actual.html', zoom_start=3)
m = create_map(reg_results['TabNet'], save_path='tabnet_predicted_vs_actual.html', zoom_start=3)

In [22]:
# --- Evaluate + compare: regression ---
reg_comparison = evaluate_models(reg_results, task_type="regression")
compare_models(
    reg_comparison,
    "REGRESSION MODEL COMPARISON (target: WEATHER_FIRE_RISK_SCORE)",
    save_path="regression_model_comparison_DNN.csv",
)


REGRESSION MODEL COMPARISON (target: WEATHER_FIRE_RISK_SCORE)
            R2     MAE    RMSE  Percentage Accuracy (100-MAPE)
Model                                                         
DNN     0.9961  0.4343  0.5817                         98.9731
TabNet  0.9561  1.7804  1.9562                         95.7676
Saved comparison table to regression_model_comparison_dnn.csv


,R2,MAE,RMSE,Percentage Accuracy (100-MAPE)
Model,,,,
DNN,0.9961,0.4343,0.5817,98.9731
TabNet,0.9561,1.7804,1.9562,95.7676
